# DarkSpot — связка веток

Читает артефакты из `darkspot_data/` (которые кладут ветки ЦА / 2GIS / ЦИАН) и собирает итоговый скоринг локаций.

> Рядом должен лежать `darkspot_integration.py`.

## Конфиг

In [2]:
import sys; sys.path.append(".")
import numpy as np
import pandas as pd
import darkspot_integration as ds

BUSINESS_TYPE      = "coffeeshop"              # coffeeshop / nail_salon / pickup_point
TARGET_AUDIENCE    = ["students", "office_workers"]
ASSUMED_AREA_SQM   = 60
BUDGET_RENT_MONTH  = 350_000

fmt = ds.FORMATS[BUSINESS_TYPE]
print(fmt["ru"], "|", ", ".join(TARGET_AUDIENCE), "|", ASSUMED_AREA_SQM, "м²")

Кофейня | students, office_workers | 60 м²


In [4]:
df_ca   = pd.read_csv("darkspot_ca.csv")
df_comp = pd.read_csv("competitors.csv")
df_cian = pd.read_csv("CianCommercialParser_raw.csv")

ds.export_demand(df_ca)
ds.export_competitors(df_comp)
ds.export_realestate(df_cian)

[export_demand] 119 районов → darkspot_data/demand.csv
[export_competitors] 70 районов → darkspot_data/competitors.csv
[export_realestate] 123 районов → darkspot_data/realestate_district.csv; 9600 объявлений → darkspot_data/realestate_listings.csv


('darkspot_data/realestate_district.csv',
 'darkspot_data/realestate_listings.csv')

## Сборка сводной таблицы

In [5]:
master = ds.build_master(BUSINESS_TYPE)
master[["district","okrug","population","avg_salary","demand",
        "n_competitors","avg_comp_rating","median_rent_sqm","avg_metro_min"]].head(10)

[build_master] районов с полными данными: 115 / 119


,district,okrug,population,avg_salary,demand,n_competitors,avg_comp_rating,median_rent_sqm,avg_metro_min
0,Академический,ЮЗАО,114228,89370,30101,61.0,4.470,3166.7,7.6
1,Алексеевский,СВАО,77681,78612,18006,86.0,4.430,2368.6,10.8
2,Алтуфьевский,СВАО,55741,78612,12921,0.0,4.315,2520.6,16.0
3,Арбат,ЦАО,36246,111713,11939,103.0,4.390,37804.9,2.9
4,Аэропорт,САО,81833,86888,20966,77.0,4.520,4125.0,7.6
5,Бабушкинский,СВАО,88627,78612,20544,51.0,4.350,2540.6,7.9
6,Басманный,ЦАО,111289,111713,36659,99.0,4.450,3846.2,6.6
7,Беговой,САО,43190,86888,11065,95.0,4.440,6805.6,5.6
8,Бескудниковский,САО,79896,86888,20469,0.0,4.315,2165.0,10.2
9,Бибирево,СВАО,158797,78612,36809,54.0,4.460,1896.9,6.2


## Регрессия аренды

Модель предсказывает ставку по фундаментальным признакам района. Остаток (реальная − предсказанная) — индикатор недооценённости: там где аренда ниже модельной при сильных показателях, есть скрытая возможность.

In [9]:
master["avg_metro_min"] = master["avg_metro_min"].fillna(master["avg_metro_min"].median())
model = ds.RentModel().fit(master)
mt = model.metrics_

print(f"R²  (test):    {mt['r2']:.3f}")
print(f"MAE (test):    {mt['mae']:,.0f} руб/м²/мес")
print(f"CV  (5-fold):  {mt['cv_mean']:.2f}  {mt['cv'].round(2)}")

coef = model.coefficients()
print("\nВклад признаков (руб/м²/мес на +1 SD):")
print(coef.round(0).to_string())

R²  (test):    -0.415
MAE (test):    1,243 руб/м²/мес
CV  (5-fold):  -0.40  [-0.16  0.11 -0.41 -0.51 -1.04]

Вклад признаков (руб/м²/мес на +1 SD):
income_index       1125.0
demand             -672.0
avg_metro_min      -647.0
n_competitors       289.0
avg_comp_rating     -49.0


In [10]:
master = model.annotate(master)
master.sort_values("rent_residual").head(8)[
    ["district","okrug","demand","n_competitors","median_rent_sqm","pred_rent","rent_residual"]
]

,district,okrug,demand,n_competitors,median_rent_sqm,pred_rent,rent_residual
27,Замоскворечье,ЦАО,18214,102.0,3599.4,6966.3,-3366.9
38,Красносельский,ЦАО,15103,107.0,3960.7,7179.7,-3219.0
53,Мещанский,ЦАО,18326,100.0,4414.0,7497.1,-3083.1
92,Тверской,ЦАО,24947,104.0,4160.6,6953.0,-2792.4
87,Сокол,САО,14777,87.0,2083.3,4858.2,-2774.9
96,Тропарёво-Никулино,ЗАО,37487,83.0,2302.1,4835.3,-2533.2
46,Ломоносовский,ЮЗАО,22938,74.0,1343.3,3812.4,-2469.1
42,Куркино,СЗАО,8937,0.0,1875.0,4328.8,-2453.8


## Скоринг районов

`Opportunity = 0.40·спрос + 0.30·(1 − конкуренция) + 0.30·недооценённость`

In [11]:
ranking = ds.score_opportunities(master, budget_rent_month=BUDGET_RENT_MONTH, area_sqm=ASSUMED_AREA_SQM)
ranking.index = range(1, len(ranking) + 1)
ranking[["district","okrug","demand","n_competitors","median_rent_sqm",
         "rent_residual","opportunity_score","fits_budget"]].head(12)

,district,okrug,demand,n_competitors,median_rent_sqm,rent_residual,opportunity_score,fits_budget
1,Выхино-Жулебино,ЮВАО,50051,0.0,1518.5,1734.7,100.0,True
2,Кунцево,ЗАО,44344,18.0,2704.4,-170.3,90.8,True
3,Гольяново,ВАО,36171,0.0,1083.3,-500.4,89.8,True
4,Тёплый Стан,ЮЗАО,34758,0.0,2200.0,-498.5,88.6,True
5,Бирюлёво Восточное,ЮАО,33531,0.0,1291.6,-854.9,87.8,True
6,Филёвский Парк,ЗАО,32907,0.0,2916.7,-574.3,87.0,True
7,Чертаново Южное,ЮАО,33516,0.0,1200.0,89.5,86.8,True
8,Ясенево,ЮЗАО,46513,38.0,2745.0,374.9,85.4,True
9,Зюзино,ЮЗАО,32489,0.0,3600.0,495.3,85.4,True
10,Митино,СЗАО,53195,59.0,2594.0,750.6,84.0,True


## Конкретные адреса

In [12]:
_, listings = ds.load_realestate()
spots = ds.find_spots(ranking, listings, area_sqm=ASSUMED_AREA_SQM,
                      budget_rent_month=BUDGET_RENT_MONTH, top_districts=6, top_n=10)
spots.index = range(1, len(spots) + 1)
display(spots[["district","address","object_type","area_sqm","price_sqm_month",
               "price_month","metro_walk_min","opportunity_score","spot_score"]]
        .style.format({"price_month": "{:,.0f} ₽", "price_sqm_month": "{:,.0f} ₽/м²",
                       "area_sqm": "{:.0f} м²",    "metro_walk_min":  "{:.0f} мин"}))

,district,address,object_type,area_sqm,price_sqm_month,price_month,metro_walk_min,opportunity_score,spot_score
1,Выхино-Жулебино,"Жулебинский, 36К1",Торговая площадь,113 м²,"1,327 ₽/м²","150,000 ₽",5 мин,100.000000,100.000000
2,Выхино-Жулебино,"Ферганская, 12",Торговая площадь,55 м²,"2,909 ₽/м²","160,000 ₽",7 мин,100.000000,88.100000
3,Выхино-Жулебино,"Ферганская, 12",Торговая площадь,110 м²,"2,991 ₽/м²","329,000 ₽",7 мин,100.000000,87.700000
4,Кунцево,"Молодогвардейская, 62к1с3",Склад,30 м²,663 ₽/м²,"19,900 ₽",nan мин,90.800000,57.800000
5,Гольяново,"Щелковское, 3С1",Склад,70 м²,"1,200 ₽/м²","84,000 ₽",7 мин,89.800000,53.500000
6,Гольяново,"1-й Иртышский, 4С3",Склад,87 м²,709 ₽/м²,"62,000 ₽",nan мин,89.800000,53.200000
7,Гольяново,"1-й Иртышский, 4С3",Склад,87 м²,709 ₽/м²,"62,000 ₽",nan мин,89.800000,53.200000
8,Кунцево,1к1,Торговая площадь,91 м²,"1,758 ₽/м²","160,000 ₽",nan мин,90.800000,51.700000
9,Гольяново,"Иркутская, 11/17",Склад,108 м²,"1,018 ₽/м²","110,000 ₽",nan мин,89.800000,51.500000
10,Гольяново,"Иркутская, 11/17",Склад,108 м²,"1,018 ₽/м²","110,000 ₽",nan мин,89.800000,51.500000


## Визуализация

In [13]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# топ районов
top15 = ranking.head(15).iloc[::-1]
fig = px.bar(top15, x="opportunity_score", y="district", orientation="h",
             color="opportunity_score", color_continuous_scale="Viridis",
             title=f"DarkSpot: топ районов — {fmt['ru']}",
             labels={"opportunity_score": "Opportunity Score", "district": ""},
             hover_data=["okrug","demand","n_competitors","median_rent_sqm"])
fig.update_layout(plot_bgcolor="white", height=520, coloraxis_showscale=False)
fig.show()

In [14]:
# спрос vs конкуренция — хочется правый нижний угол
fig = px.scatter(ranking, x="demand", y="n_competitors",
                 size="median_rent_sqm", color="opportunity_score",
                 color_continuous_scale="RdYlGn", size_max=26, text="district",
                 hover_data=["okrug","median_rent_sqm","rent_residual"],
                 title="Спрос vs конкуренция",
                 labels={"demand": "ЦА", "n_competitors": "Конкурентов",
                         "opportunity_score": "Opportunity"})
fig.update_traces(textposition="top center", textfont_size=8)
fig.update_layout(plot_bgcolor="white", height=560)
fig.show()

In [15]:
# диагностика регрессии
fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Предсказанная vs реальная аренда", "Вклад признаков"))

lo = min(master["median_rent_sqm"].min(), master["pred_rent"].min())
hi = max(master["median_rent_sqm"].max(), master["pred_rent"].max())

opp_colors = ranking.set_index("district")["opportunity_score"].reindex(master["district"]).values
fig.add_trace(go.Scatter(x=master["median_rent_sqm"], y=master["pred_rent"],
                         mode="markers", text=master["district"],
                         marker=dict(color=opp_colors, colorscale="Viridis", size=8),
                         name="районы"), row=1, col=1)
fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode="lines",
                         line=dict(dash="dash", color="red"), name="идеал"), row=1, col=1)

cs = coef.sort_values()
colors = ["#d62728" if v < 0 else "#2ca02c" for v in cs.values]
fig.add_trace(go.Bar(x=cs.values, y=cs.index, orientation="h", marker_color=colors), row=1, col=2)

fig.update_xaxes(title_text="Реальная ставка", row=1, col=1)
fig.update_yaxes(title_text="Предсказанная",   row=1, col=1)
fig.update_layout(height=460, showlegend=False, plot_bgcolor="white",
                  title_text=f"R²={mt['r2']:.2f}, MAE={mt['mae']:,.0f} руб/м²")
fig.show()

In [16]:
import folium

def opp_color(s):
    for threshold, color in [(70,"#1a9850"),(50,"#91cf60"),(35,"#fee08b"),(20,"#fc8d59")]:
        if s >= threshold: return color
    return "#d73027"

m = folium.Map(location=[ranking["lat"].mean(), ranking["lon"].mean()],
               zoom_start=11, tiles="cartodbpositron")

for _, r in ranking.iterrows():
    folium.CircleMarker(
        location=[r["lat"], r["lon"]],
        radius=8 + r["opportunity_score"] / 8,
        color=opp_color(r["opportunity_score"]), fill=True, fill_opacity=0.65,
        popup=folium.Popup(
            f"<b>{r['district']}</b> ({r['okrug']})<br>"
            f"Opportunity: <b>{r['opportunity_score']}</b><br>"
            f"ЦА: {r['demand']:,.0f} | конкурентов: {r['n_competitors']:.0f}<br>"
            f"Аренда: {r['median_rent_sqm']:,.0f} ₽/м² | остаток: {r['rent_residual']:,.0f}",
            max_width=240)
    ).add_to(m)

spots_geo = spots.merge(ranking[["district","lat","lon"]], on="district", how="left")
for _, s in spots_geo.head(5).iterrows():
    folium.Marker(
        location=[s["lat"] + np.random.uniform(-0.004, 0.004),
                  s["lon"] + np.random.uniform(-0.004, 0.004)],
        icon=folium.Icon(color="darkblue", icon="star"),
        popup=folium.Popup(
            f"<b>{s['address']}</b><br>{s['district']}<br>"
            f"{s['object_type']}, {s['area_sqm']:.0f} м²<br>"
            f"{s['price_month']:,.0f} ₽/мес | spot score: <b>{s['spot_score']}</b>",
            max_width=260)
    ).add_to(m)

m

## Итоговая рекомендация

In [ ]:
bd, bs = ranking.iloc[0], spots.iloc[0]

print(f"{fmt['ru']} | ЦА: {', '.join(TARGET_AUDIENCE)} | бюджет: {BUDGET_RENT_MONTH:,.0f} ₽/мес\n")
print(f"Район:   {bd['district']} ({bd['okrug']})")
print(f"  Opportunity:  {bd['opportunity_score']}/100")
print(f"  ЦА:           {bd['demand']:,.0f} чел.")
print(f"  Конкурентов:  {bd['n_competitors']:.0f}")
print(f"  Аренда:       {bd['median_rent_sqm']:,.0f} ₽/м²/мес")
print(f"  Недооценка:   {bd['rent_residual']:,.0f} ₽/м²\n")
print(f"Объект:  {bs['address']}, {bs['district']}")
print(f"  {bs['object_type']}, {bs['area_sqm']:.0f} м²")
print(f"  {bs['price_month']:,.0f} ₽/мес ({bs['price_sqm_month']:,.0f} ₽/м²)")
print(f"  до метро {bs['metro_walk_min']:.0f} мин | spot score {bs['spot_score']}/100")